# Plan for today:
- PyTorch installation
- PyTorch basics
- Data loading and simple models in PyTorch (linear regression, images of digits)

# Install pytorch and torchvision
Installation instruction: https://pytorch.org/  
e.g. with conda:  
`conda install pytorch torchvision -c pytorch`  
  
Different commands for GPU version -> all in PyTorch website

# PyTorch resources
- https://pytorch.org/tutorials/
- https://github.com/yunjey/pytorch-tutorial
- http://cs231n.stanford.edu/

# PyTorch basics

## Tensors

In [4]:
import torch

Basic datatype in PyTorch is a Tensor (syntax very similar to numpy array). https://pytorch.org/docs/stable/tensors.html  
**A `torch.Tensor` is a multi-dimensional matrix containing elements of a single data type.**

In [5]:
data = torch.tensor([1, 2, 3])
print(data)
print(data.size())
print(data.dtype)

tensor([1, 2, 3])
torch.Size([3])
torch.int64


In [6]:
data = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print(data)
print(data.size())

tensor([[1., 2., 3.],
        [4., 5., 6.]])
torch.Size([2, 3])


In [7]:
torch.zeros((3, 4))

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [8]:
torch.ones((3, 4))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [9]:
# Random numbers
torch.rand(3, 5)

tensor([[0.3643, 0.5543, 0.8821, 0.4887, 0.4397],
        [0.4693, 0.5540, 0.6312, 0.3383, 0.7263],
        [0.1331, 0.7373, 0.1621, 0.6878, 0.2067]])

## CPU vs GPU (CUDA)
Tensors can live on CPU or on GPU if torch is installed with Nvidia CUDA support. *CUDA tensors allow for much faster computation!*

In [10]:
torch.cuda.is_available()

False

To make code compatible with both GPU and CPU, you can set the device at the beginning of your code and always transfer your tensors/models to this device

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
print(device)

cpu


In [18]:
a = torch.tensor(5, device=device) # Create a tensor on a given device
print(a)

tensor(5)


In [19]:
a = a.to(device) # Copy a tensor to another device
print(a)

tensor(5)


In [15]:
a = a.to(torch.device('cpu'))
print(a)
print(a.device)

tensor(5)
cpu


## Computation

In [20]:
a = torch.tensor(5.0)
b = torch.tensor(2.0)
print(a, b)

tensor(5.) tensor(2.)


In [21]:
c = a + b
print(c)

tensor(7.)


In [22]:
c = a - b
print(c)

tensor(3.)


In [23]:
c = a * b
print(c)

tensor(10.)


In [24]:
c = a / b
print(c)

tensor(2.5000)


In [25]:
# Matrix computations
a = torch.rand(3, 4)
b = torch.rand(4, 1)
print(a)
print(b)

tensor([[0.4327, 0.4560, 0.4827, 0.2511],
        [0.3895, 0.3684, 0.4276, 0.8762],
        [0.2755, 0.4564, 0.2474, 0.9318]])
tensor([[0.4911],
        [0.3529],
        [0.2730],
        [0.8861]])


In [26]:
c = a.matmul(b)
print(c)

tensor([[0.7277],
        [1.2144],
        [1.1895]])


In [27]:
print(c + torch.rand(3, 1))

tensor([[0.7792],
        [2.0839],
        [1.7868]])


# PyTorch computational graph
Important property of PyTorch tensor operations - they create a **dynamic computational graph**.  
The tensor **c** knows that how it was created from **a** and **b**.  
Thanks to this, derivatives (gradients) can be computed for tensors.  
Example:  

$
a = 2 \\
b = 1 \\
c = a + b \\
d = b + 1 \\
e = c * d
$

<img src="tree-eval.png" alt="Drawing" style="width: 600px;"/>  
*Source: http://colah.github.io/posts/2015-08-Backprop/ - good blog post to understand backpropagation*

### Compute for a=2, b=1

In [68]:
# Create a and b
a = torch.tensor(2.0, requires_grad=True) # if no tensors require gradient, the computational graph won't be created
b = torch.tensor(1.0, requires_grad=True)
# Compute c, d, e
c = a + b
d = b + 1
e = c * d
print('a:', a)
print('b:', b)
print('c:', c)
print('d:', d)
print('e:', e)

a: tensor(2., requires_grad=True)
b: tensor(1., requires_grad=True)
c: tensor(3., grad_fn=<AddBackward0>)
d: tensor(2., grad_fn=<AddBackward0>)
e: tensor(6., grad_fn=<MulBackward0>)


In [69]:
# grad_fn stores information about computation
print(c.grad_fn)
print(e.grad_fn)

### Now we can compute the derivative of $e$ w.r.t. $a, b, c, d$
$
\frac {\partial e} {\partial d}, \frac {\partial e} {\partial c}, \frac {\partial e} {\partial b}, \frac {\partial e} {\partial a} \\
$
<img src="tree-eval-derivs.png" alt="Drawing" style="width: 600px;"/>  

$
e = c * d \\
\frac {\partial e} {\partial c} = d = 2 \\
\frac {\partial e} {\partial d} = c = 3 \\
\frac {\partial e} {\partial a} = \frac {\partial e} {\partial c} \frac {\partial c} {\partial a} = 2 \\
\frac {\partial e} {\partial b} = \frac {\partial e} {\partial c} \frac {\partial c} {\partial b} + \frac {\partial e} {\partial d} \frac {\partial d} {\partial b} = 2 * 1 + 3 * 1 = 5
$

Compute gradients with `torch.autograd.grad` https://pytorch.org/docs/stable/autograd.html

In [70]:
# Compute gradients with torch.autograd.grad https://pytorch.org/docs/stable/autograd.html

In [71]:
grad_a, grad_b, grad_c, grad_d = torch.autograd.grad(e, [a, b, c, d])
# ^ running this frees the computational graph by default, but we can specify retain_graph=True
# if we need to compute the gradients again

In [72]:
print(grad_a, grad_b, grad_c, grad_d)

tensor(2.) tensor(5.) tensor(2.) tensor(3.)


In [73]:
# Another way for leaf tensors with `backward` method

In [74]:
# Create the computational graph again
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
c = a + b
d = b + 1
e = c * d
print(a, b, c, d, e)

tensor(2., requires_grad=True) tensor(1., requires_grad=True) tensor(3., grad_fn=<AddBackward0>) tensor(2., grad_fn=<AddBackward0>) tensor(6., grad_fn=<MulBackward0>)


In [75]:
e.backward() # Propagates the gradient to the leaf tensors

In [32]:
print(a.grad, b.grad, c.grad, d.grad)

tensor(2.) tensor(5.) None None
